# 05 - Employee Attrition Risk Scoring System

## Objective

Convert machine learning predictions into employee risk scores and risk categories that HR teams can use for proactive retention planning.

## Business Goal

Instead of predicting only Attrition = Yes / No,
generate:

- Attrition Probability
- Risk Score
- Risk Level (Low / Medium / High)

This output will become the input dataset for Power BI dashboards and business reporting.

## 1. Load Dataset

In [52]:
import pandas as pd

df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 2. Recreate Modeling Dataset

In [53]:
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Attrition"])
y = df["Attrition"]

X = pd.get_dummies(X, drop_first=True)

## 3. Train-Test Split

In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 4. Scale Features for Logistic Regression

In [55]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Train Final Logistic Regression Model

In [56]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

lr.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## 6. Generate Attrition Probabilities

In [57]:
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

y_prob[:10]

array([0.29293576, 0.02032816, 0.30993067, 0.04876736, 0.65156982,
       0.67094635, 0.07806038, 0.21161841, 0.01983734, 0.24541766])

## 7. Create Employee Risk Levels

In [58]:
risk_df = X_test.copy()

risk_df["Actual_Attrition"] = y_test.values

risk_df["Attrition_Probability"] = y_prob

risk_df.head()

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes,Actual_Attrition,Attrition_Probability
1061,24,830,13,2,1,1495,4,78,3,1,...,False,False,False,False,True,True,False,False,0,0.292936
891,44,1117,2,1,1,1246,1,72,4,1,...,False,False,True,False,False,True,False,False,0,0.020328
456,31,688,7,3,1,613,3,44,2,3,...,False,False,False,False,False,False,False,False,0,0.309931
922,44,1199,4,2,1,1288,3,92,4,5,...,False,False,False,False,False,False,False,False,0,0.048767
69,36,318,9,3,1,90,4,79,2,1,...,False,False,True,False,False,True,False,True,1,0.651570


### Define Risk Categories

Low Risk    : 0.00 - 0.30

Medium Risk : 0.30 - 0.60

High Risk   : 0.60 - 1.00

In [59]:
def assign_risk(prob):

    if prob < 0.30:
        return "Low"

    elif prob < 0.60:
        return "Medium"

    else:
        return "High"


risk_df["Risk_Level"] = risk_df["Attrition_Probability"].apply(assign_risk)

risk_df.head()

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes,Actual_Attrition,Attrition_Probability,Risk_Level
1061,24,830,13,2,1,1495,4,78,3,1,...,False,False,False,True,True,False,False,0,0.292936,Low
891,44,1117,2,1,1,1246,1,72,4,1,...,False,True,False,False,True,False,False,0,0.020328,Low
456,31,688,7,3,1,613,3,44,2,3,...,False,False,False,False,False,False,False,0,0.309931,Medium
922,44,1199,4,2,1,1288,3,92,4,5,...,False,False,False,False,False,False,False,0,0.048767,Low
69,36,318,9,3,1,90,4,79,2,1,...,False,True,False,False,True,False,True,1,0.651570,High


## 8. Risk Distribution Analysis

In [60]:
risk_df["Risk_Level"].value_counts()

Risk_Level
Low       161
Medium     71
High       62
Name: count, dtype: int64

In [61]:
risk_df["Risk_Level"].value_counts(normalize=True) * 100

Risk_Level
Low       54.761905
Medium    24.149660
High      21.088435
Name: proportion, dtype: float64

## 9. Department Risk Analysis

Analyze risk distribution across departments to identify business units requiring retention intervention.

In [62]:
department_info = df.loc[X_test.index, ["Department"]]

risk_df = risk_df.merge(
    department_info,
    left_index=True,
    right_index=True
)

risk_df.head()

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes,Actual_Attrition,Attrition_Probability,Risk_Level,Department
1061,24,830,13,2,1,1495,4,78,3,1,...,False,False,True,True,False,False,0,0.292936,Low,Sales
891,44,1117,2,1,1,1246,1,72,4,1,...,True,False,False,True,False,False,0,0.020328,Low,Research & Development
456,31,688,7,3,1,613,3,44,2,3,...,False,False,False,False,False,False,0,0.309931,Medium,Sales
922,44,1199,4,2,1,1288,3,92,4,5,...,False,False,False,False,False,False,0,0.048767,Low,Research & Development
69,36,318,9,3,1,90,4,79,2,1,...,True,False,False,True,False,True,1,0.651570,High,Research & Development


In [63]:
pd.crosstab(
    risk_df["Department"],
    risk_df["Risk_Level"],
    normalize="index"
) * 100

Risk_Level,High,Low,Medium
Department,,,
Human Resources,20.000000,53.333333,26.666667
Research & Development,16.751269,59.390863,23.857868
Sales,31.707317,43.902439,24.390244


### Department Risk Insights

Observe which departments contain the highest percentage of High Risk employees.

## 10. Job Role Risk Analysis

Identify job roles with elevated attrition risk.

In [64]:
jobrole_info = df.loc[X_test.index, ["JobRole"]]

risk_df = risk_df.merge(
    jobrole_info,
    left_index=True,
    right_index=True
)

risk_df.head()

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes,Actual_Attrition,Attrition_Probability,Risk_Level,Department,JobRole
1061,24,830,13,2,1,1495,4,78,3,1,...,False,True,True,False,False,0,0.292936,Low,Sales,Sales Representative
891,44,1117,2,1,1,1246,1,72,4,1,...,False,False,True,False,False,0,0.020328,Low,Research & Development,Research Scientist
456,31,688,7,3,1,613,3,44,2,3,...,False,False,False,False,False,0,0.309931,Medium,Sales,Manager
922,44,1199,4,2,1,1288,3,92,4,5,...,False,False,False,False,False,0,0.048767,Low,Research & Development,Manager
69,36,318,9,3,1,90,4,79,2,1,...,False,False,True,False,True,1,0.651570,High,Research & Development,Research Scientist


In [65]:
pd.crosstab(
    risk_df["JobRole"],
    risk_df["Risk_Level"],
    normalize="index"
) * 100

Risk_Level,High,Low,Medium
JobRole,,,
Healthcare Representative,4.347826,65.217391,30.434783
Human Resources,27.272727,45.454545,27.272727
Laboratory Technician,25.000000,38.636364,36.363636
Manager,5.263158,84.210526,10.526316
Manufacturing Director,12.500000,65.625000,21.875000
Research Director,0.000000,100.000000,0.000000
Research Scientist,22.222222,54.166667,23.611111
Sales Executive,22.641509,52.830189,24.528302
Sales Representative,60.869565,13.043478,26.086957


### Job Role Risk Insights

Identify job roles requiring targeted retention programs.

In [66]:
pd.crosstab(
    risk_df["Department"],
    risk_df["Risk_Level"],
    normalize="index"
) * 100

Risk_Level,High,Low,Medium
Department,,,
Human Resources,20.000000,53.333333,26.666667
Research & Development,16.751269,59.390863,23.857868
Sales,31.707317,43.902439,24.390244


In [67]:
pd.crosstab(
    risk_df["JobRole"],
    risk_df["Risk_Level"],
    normalize="index"
) * 100

Risk_Level,High,Low,Medium
JobRole,,,
Healthcare Representative,4.347826,65.217391,30.434783
Human Resources,27.272727,45.454545,27.272727
Laboratory Technician,25.000000,38.636364,36.363636
Manager,5.263158,84.210526,10.526316
Manufacturing Director,12.500000,65.625000,21.875000
Research Director,0.000000,100.000000,0.000000
Research Scientist,22.222222,54.166667,23.611111
Sales Executive,22.641509,52.830189,24.528302
Sales Representative,60.869565,13.043478,26.086957


## Key Business Findings

### Department-Level Findings

- Sales department has the highest concentration of High Risk employees.
- Research & Development shows the lowest attrition risk profile.

### Job Role Findings

- Sales Representatives exhibit the highest attrition risk.
- Laboratory Technicians and Human Resources roles also show elevated risk levels.
- Managers and Research Directors demonstrate strong retention profiles.

### HR Recommendation

Prioritize retention initiatives for:

1. Sales Representatives
2. Sales Executives
3. Laboratory Technicians
4. Human Resources employees

Focus areas:
- Compensation review
- Career progression
- Work-life balance
- Manager engagement

## 11. Export Final Risk Scoring Dataset

In [69]:
risk_df.columns

Index(['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
       'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Research & Development', 'Department_Sales',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'Gender_Male',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
  

## 12. Create Business-Friendly Risk Dataset

In [70]:
business_df = df.loc[X_test.index, [
    "EmployeeNumber",
    "Department",
    "JobRole",
    "Age",
    "MonthlyIncome",
    "YearsAtCompany"
]].copy()

business_df["Actual_Attrition"] = y_test.values

business_df["Attrition_Probability"] = y_prob

business_df["Risk_Level"] = business_df[
    "Attrition_Probability"
].apply(assign_risk)

business_df.head()

,EmployeeNumber,Department,JobRole,Age,MonthlyIncome,YearsAtCompany,Actual_Attrition,Attrition_Probability,Risk_Level
1061,1495,Sales,Sales Representative,24,2033,1,0,0.292936,Low
891,1246,Research & Development,Research Scientist,44,2011,10,0,0.020328,Low
456,613,Sales,Manager,31,11557,5,0,0.309931,Medium
922,1288,Research & Development,Manager,44,19190,25,0,0.048767,Low
69,90,Research & Development,Research Scientist,36,3388,1,1,0.651570,High


In [71]:
business_df.info()

<class 'pandas.DataFrame'>
Index: 294 entries, 1061 to 285
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   EmployeeNumber         294 non-null    int64  
 1   Department             294 non-null    str    
 2   JobRole                294 non-null    str    
 3   Age                    294 non-null    int64  
 4   MonthlyIncome          294 non-null    int64  
 5   YearsAtCompany         294 non-null    int64  
 6   Actual_Attrition       294 non-null    int64  
 7   Attrition_Probability  294 non-null    float64
 8   Risk_Level             294 non-null    str    
dtypes: float64(1), int64(5), str(3)
memory usage: 23.0 KB


In [72]:
business_df.head(10)

,EmployeeNumber,Department,JobRole,Age,MonthlyIncome,YearsAtCompany,Actual_Attrition,Attrition_Probability,Risk_Level
1061,1495,Sales,Sales Representative,24,2033,1,0,0.292936,Low
891,1246,Research & Development,Research Scientist,44,2011,10,0,0.020328,Low
456,613,Sales,Manager,31,11557,5,0,0.309931,Medium
922,1288,Research & Development,Manager,44,19190,25,0,0.048767,Low
69,90,Research & Development,Research Scientist,36,3388,1,1,0.651570,High
1164,1641,Research & Development,Manufacturing Director,40,7945,4,0,0.670946,High
406,543,Research & Development,Manufacturing Director,52,7969,5,0,0.078060,Low
1330,1866,Research & Development,Manager,43,19392,16,0,0.211618,Low
1232,1728,Research & Development,Manufacturing Director,35,6883,7,0,0.019837,Low
1311,1839,Research & Development,Research Scientist,18,1514,0,0,0.245418,Low


## 13. Export Risk Scoring Dataset

In [76]:
business_df.to_csv(
    "../data/cleaned/employee_attrition_risk_scores.csv",
    index=False
)

print("Risk scoring dataset exported successfully.")

Risk scoring dataset exported successfully.
